# 01 · ViT CIFAR-10 —— 把图切成“字”，与 CNN 同台看归纳偏置

**家族位置**：`06_Transformer_Vision_Multimodal` 第 1 站。05 让机器“读书”（字→字），本章让机器“看图”：32×32 图切成 4×4=64 个 patch 块当“字”，加 `[CLS]` 通读打分——这就是 ViT。与 `SmallCNN`（3 层卷积+GAP，0.4M）同 CIFAR-10 子集（train 8000/val 1000/test 全量）同 6ep 同台，看“无卷积”要付出多少数据代价。无外网（02 缓存），CPU 约 12 分钟。

**学习目标**
1. patch embedding：Conv2d(patch,stride=patch) 一步切块+投影，64 个 patch token
2. 可学习位置编码（含 CLS 位 65 长）为什么必须，vs 05 的 sin-cos
3. 小数据上 ViT vs CNN：同预算谁赢，为什么（归纳偏置：平移等变/局部性）
4. CLS 注意力可视化：汇总位在看哪些 patch

## 1. 原理：从“字序列”到“图块序列”

### 通俗理解

**一句话**：05 的 Transformer 读“字”，ViT 把图切成 64 块小拼图，每块当一个“字”，前面再加 1 个 `[CLS]` 当班长——65 个 token 通读 6 层，最后班长打分分类。

**比喻**：CNN 像用放大镜逐格看（局部+平移等变，天生懂“猫在哪都是猫”）；ViT 像把拼图全摊开同时看——没有放大镜，必须靠数据自己学“相邻块有关”，所以小数据吃亏、大数据反超。

### 结构账

```
输入：  x (B,3,32,32) → patchify Conv2d(3→192, k=4,s=4) → (B,64,192) patch tokens
加长：  [CLS](1,192) 拼前面 → 65 长 → + 可学习 pos_embed(65,192)
Encoder×6： x = LN(x+MHA(x)) → LN(x+MLP×4)   双向（分类无因果）
分类头： logits = Linear(h[CLS])   (192 → 10)
对照：  SmallCNN 3层3×3+BN+GAP 0.4M  vs  ViT-Tiny d192/L6/h3 1.6M
训练：  CE，AdamW 3e-4 wd 0.05，batch 128，12ep，seed=0
```

- **与 05-02 BERT 的对应**：同 Encoder-only 双向+CLS，差别只是 token 从“字嵌入”换成“patch 投影”
- **评估**：val/test acc + patch 网格 + CLS 注意力热力 + 错例

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import load_cifar10_local, CIFAR10_CLASSES
from common.models import ViTTiny, SmallCNN
from common.engine import fit, run_epoch
from common.utils import set_seed, setup_chinese_font, count_params

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

N_TRAIN, N_VAL = 4000, 800
EPOCHS, BATCH, LR = 6, 128, 3e-4
Xtr, ytr, Xva, yva, Xte, yte = load_cifar10_local(N_TRAIN, N_VAL, seed=0)
print(f"train {tuple(Xtr.shape)} / val {tuple(Xva.shape)} / test {tuple(Xte.shape)} | 类 {CIFAR10_CLASSES}")


## 2. 数据：CIFAR-10 子集（02 缓存，无外网）

train 8000 / val 1000 / test 10000 全量；标准 0.49/0.48/0.44 归一；子集随机种子固定可复现。

In [ ]:
# fig0：8 张样例 + patch 网格（第 1 张切 4×4 patch 示意）
mean = np.array([0.4914, 0.4822, 0.4465]).reshape(3,1,1)
std = np.array([0.2470, 0.2435, 0.2616]).reshape(3,1,1)
def unnorm(x):
    return np.clip((x.numpy()*std+mean).transpose(1,2,0), 0, 1)
fig, axes = plt.subplots(2, 4, figsize=(9, 4.6))
for ax, i in zip(axes[0], range(8)):
    ax.imshow(unnorm(Xtr[i]))
    ax.set_title(CIFAR10_CLASSES[int(ytr[i])], fontsize=9)
    ax.axis("off")
# patch 网格画在第 1 张放大
ax_big = axes[1][1]
ax_big.imshow(unnorm(Xtr[0]))
for k in range(1, 8):
    ax_big.axhline(k*4-0.5, color="yellow", linewidth=0.8)
    ax_big.axvline(k*4-0.5, color="yellow", linewidth=0.8)
ax_big.set_title("4×4 patch 网格（64 块）", fontsize=9)
ax_big.axis("off")
for ax in [axes[1][0], axes[1][2], axes[1][3]]:
    ax.axis("off")
axes[1][0].text(0.5, 0.5, "32×32 → 64 个\n4×4 patch\n+1 CLS=65 token", ha="center", va="center", fontsize=10)
axes[1][2].text(0.5, 0.5, "CNN: 放大镜逐格看\nViT: 拼图全摊开看", ha="center", va="center", fontsize=10)
axes[1][3].text(0.5, 0.5, f"train {len(Xtr)}", ha="center", va="center", fontsize=10)
plt.suptitle("CIFAR-10 子集与 patch 切分", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig0_patches.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. 同台：ViT-Tiny vs SmallCNN（同数据同预算）

In [ ]:
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, yva), batch_size=512)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=512)

vit = ViTTiny(dim=128, depth=4, heads=4)
cnn = SmallCNN()
print(f"ViT params={count_params(vit)} | CNN params={count_params(cnn)}")

torch.manual_seed(0)
hist_vit = fit(vit, train_loader, val_loader, epochs=EPOCHS, lr=LR)
print("--- CNN ---")
torch.manual_seed(0)
hist_cnn = fit(cnn, train_loader, val_loader, epochs=EPOCHS, lr=1e-3,
               optimizer_cls=torch.optim.Adam, weight_decay=0.0)

# test
for name, model in [("ViT", vit), ("CNN", cnn)]:
    te_loss, te_acc = run_epoch(model, test_loader, nn.CrossEntropyLoss(), None)
    va_acc = max(hist_vit["val_acc"] if name=="ViT" else hist_cnn["val_acc"])
    print(f"{name} best-val={va_acc:.4f} test={te_acc:.4f}")

# fig1：val acc 双曲线 + train loss
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(hist_vit["val_acc"], label="ViT", color="#4C72B0")
axes[0].plot(hist_cnn["val_acc"], label="CNN", color="#DD8452")
axes[0].set_title("val acc（同子集同 6ep）")
axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(hist_vit["train_loss"], label="ViT", color="#4C72B0")
axes[1].plot(hist_cnn["train_loss"], label="CNN", color="#DD8452")
axes[1].set_title("train loss")
axes[1].set_xlabel("epoch"); axes[1].legend()
plt.suptitle("ViT vs CNN（小数据同台）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：test 柱状
te_vit = run_epoch(vit, test_loader, nn.CrossEntropyLoss(), None)[1]
te_cnn = run_epoch(cnn, test_loader, nn.CrossEntropyLoss(), None)[1]
fig, ax = plt.subplots(figsize=(4.8, 3.6))
ax.bar(["CNN 0.4M", "ViT 1.6M"], [te_cnn, te_vit], color=["#DD8452","#4C72B0"])
ax.set_ylim(0,1.0); ax.set_ylabel("test acc")
for i, v in enumerate([te_cnn, te_vit]):
    ax.text(i, v+0.02, f"{v:.3f}", ha="center", fontsize=10)
ax.set_title("CIFAR-10 子集 test：归纳偏置的价值")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 可视化：CLS 在看哪 + 错例

In [ ]:
# fig3：CLS 注意力热力（测试集第 1 张，最后一层头平均，8×8 patch）
vit.eval()
with torch.no_grad():
    x = Xte[:1]
    logits, attns = vit(x, return_attn=True)
    last = attns[-1][0].mean(dim=0).cpu().numpy()  # (65,65)
    cls_row = last[0, 1:].reshape(8, 8)  # CLS 对 64 patch
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].imshow(unnorm(Xte[0]))
axes[0].set_title(f"原图: {CIFAR10_CLASSES[int(yte[0])]}", fontsize=10)
axes[0].axis("off")
im = axes[1].imshow(cls_row, cmap="viridis")
axes[1].set_title("CLS 注意力（层6头平均，8×8 patch）", fontsize=10)
axes[1].set_xlabel("patch 列"); axes[1].set_ylabel("patch 行")
plt.colorbar(im, ax=axes[1], shrink=0.8)
plt.tight_layout()
plt.savefig(FIGS / "fig3_cls_attn.png", dpi=150, bbox_inches="tight")
plt.show()

# fig4：ViT 错例 4 张（test 前 200 内）
vit.eval()
with torch.no_grad():
    xe = Xte[:200]; ye = yte[:200]
    pred = vit(xe).argmax(dim=-1)
wrong = [i for i in range(200) if int(pred[i]) != int(ye[i])][:4]
fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for ax, i in zip(axes, wrong):
    ax.imshow(unnorm(Xte[i]))
    ax.set_title(f"true {CIFAR10_CLASSES[int(yte[i])]}\npred {CIFAR10_CLASSES[int(pred[i])]}", fontsize=8, color="#C0392B")
    ax.axis("off")
plt.suptitle("ViT 错例（语义相近类为主）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig4_errors.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"ViT test={te_vit:.4f} CNN test={te_cnn:.4f}")


## 5. 总结与下一步

**本项目收获**

1. patchify+CLS+可学习 pos 三件套闭环，ViT-Tiny 1.6M 参 CPU 可训
2. 小数据同台：CNN 归纳偏置赢（见 fig2），ViT 需大数据/预训练才反超
3. CLS 注意力可视化：汇总位聚焦目标 patch
4. 衔接 05：Encoder-only 双向+CLS 与 BERT 同构，token 从字换成 patch

**下一步**：`02_CLIP_ZeroShot`（图文对比对齐，zero-shot 分类）→ `03_Multimodal_Experience`（BLIP/LLaVA 推理体验，视觉编码器+投影+LLM）。